# Training a BIT model from a neuralshed collection

**BIT** is a CTC brain-to-text decoder: neural features in, characters out, with no alignment between the two. You hand it a window of someone attempting to communicate and it emits the decoded text.

This notebook walks the whole path from *a Collection of neuralshed datasets* to *a model taking a gradient step*, one stage at a time, showing what each stage adds to the data. It runs on: `card_2024` (participant T15) and `willet_2023` (participant T12).

> **Requires the data drive.** `NEURALSHED_DATA_ROOT` must point at a mount holding `card_2024` and `willet_2023`. The first cell fails with a clear message if it does not.

## The pipeline

```
  Studies                           card_2024 · willet_2023
     │
     ▼
  CombineStudies                    fan-in of collection members
     │
     ▼
  StandardizeText                   column: canonical_text
  ValidateTarget                    column: target_is_valid
  AssignHashSplit                   column: split
     │
     ▼
  CollectionManifest                resolve Data Layouts
     │
     ▼
  validate_collection_for_model     pre-flight validation
     │
     ▼
  Segmenter + extractors            events frame -> arrays
     │
     ▼
  BitPreprocessingState             fitted on TRAIN only (means and standard deviations)
     │
     ▼
  collate_bit                       BitBatch, padded
     │
════▼══════════════════════════════ model ═════════════════════════════════════
  BitReadIn            per-layout adapters, C -> D
  BitPatcher           shared
  BitTrunk             shared
  CTC head             shared, D -> n_classes
```

Everything above the line is data engineering and serializes with the collection. Everything below it is learned. **Only `BitReadIn` knows how many channels a recording has**.

In [1]:
import os, logging, time, warnings
warnings.filterwarnings("ignore")
os.environ["TQDM_DISABLE"] = "1"     # neuralset's prepare() progress bars
logging.disable(logging.INFO)        # its extractors log per-event at INFO

import numpy as np
import pandas as pd
import torch

import neuralset as ns
from neuralshed import studies, transforms          # registers every step by name
from neuralshed.database import collections as dbc

pd.set_option("display.width", 170, "display.max_columns", 40)

PATHS = dbc.data_dirs(["card_2024", "willet_2023"], must_exist=True)
for name, path in PATHS.items():
    print(f"{name:22s} {path}")

Card2024Brain2Text     /Volumes/seagate_0/tether/datasets/card_2024/data
Willett2023Sentences   /Volumes/seagate_0/tether/datasets/willet_2023/data


---
## 0. Build a Collection

A **collection** is normally saved in the neuralshed database and loaded by name:

```python
chain = dbc.load_chain(collection_name="my-collection-N")
```

In this notebook we will compose an equivalent chain **in memory**. It is the same object `load_chain` would return.

Both members are queried down to two sessions each, which keeps the notebook light while still giving more than one session per participant, so per-session adapters have something to distinguish.

In [2]:
CARD = studies.Card2024Brain2Text(path=PATHS["Card2024Brain2Text"], query="session_index < 2")
WILLET = studies.Willett2023Sentences(path=PATHS["Willett2023Sentences"], query="session_index < 2")

print("card_2024  ", type(CARD).__name__, "| T15, attempted speech, 20 ms bins")
print("willet_2023", type(WILLET).__name__, "| T12, attempted speech, 20 ms bins")

card_2024   Card2024Brain2Text | T15, attempted speech, 20 ms bins
willet_2023 Willett2023Sentences | T12, attempted speech, 20 ms bins


---
## 1. `CombineStudies` — the fan-in

neuralset's step graph is strictly linear: a `Study` is a source, every `EventsTransform` is one-in one-out, and a `Chain` runs them in order. There is **no fan-in**, so pooling several studies has no configurable form. `CombineStudies` is that missing node.

Its one job beyond concatenation is writing **`collection_member`**. Why not use the `study` column neuralset already writes? Because two members can run the same `Study` class under different queries — sessions 1-20 as one member, 21-45 as another — and every downstream rule (which trials are targets, which split policy, is it already normalized) is *per member*. `study` cannot tell those apart.

It is set on each frame **before** the concat, because `standardize_events` back-fills only its own five invariants (`subject`, `session`, `task`, `run`, `study`). A column present on one frame and absent on another comes out `NaN`.

In [3]:
t0 = time.time()
events = transforms.CombineStudies(chains={"card_2024": CARD, "willet_2023": WILLET}).run()
events = events.reset_index(drop=True)
print(f"{len(events)} events in {time.time() - t0:.1f}s "
      f"(exca caches, so a re-run is much faster)\n")

events.groupby(["collection_member", "type"]).size().unstack(fill_value=0)

4191 events in 10.1s (exca caches, so a re-run is much faster)



type,BandPower,Sentence,SpikeCounts,Stimulus
collection_member,,,,
card_2024,706,671,706,0
willet_2023,34,680,34,1360


In [4]:
events.head(20)

,type,start,duration,timeline,subject,session,task,run,feature,n_features,unit,normalization,processing,frequency,filepath,method,threshold,lower_freq,upper_freq,text,...,study,date,post_implant_day,original_split,block,trial_id,task_condition,valid,valid_description,brain_region,context,stop,dataset_id,participant_id,collection_member,description,stimulus_kind,block_type,loop_type,code
0,SpikeCounts,0.0,6.42,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,threshold_crossings,256,counts/bin,z_scored,linear regression referencing per 64-electrode...,50.0,"{""cls"":""Card2024Brain2Text"",""kwargs"":{""stream""...",threshold_crossing,-4.5 x RMS,NaN,NaN,NaN,...,Card2024Brain2Text,2023-08-11,25,train,2,0.0,attempted,True,,,NaN,6.42,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
1,BandPower,0.0,6.42,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,spike_band_power,256,uV^2,z_scored,linear regression referencing per 64-electrode...,50.0,"{""cls"":""Card2024Brain2Text"",""kwargs"":{""stream""...",NaN,NaN,250.0,5000.0,NaN,...,Card2024Brain2Text,2023-08-11,25,train,2,0.0,attempted,True,,,NaN,6.42,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
2,Sentence,0.0,6.42,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bring it closer.,...,Card2024Brain2Text,2023-08-11,25,train,2,0.0,attempted,True,,NaN,,6.42,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
3,SpikeCounts,0.0,9.62,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,threshold_crossings,256,counts/bin,z_scored,linear regression referencing per 64-electrode...,50.0,"{""cls"":""Card2024Brain2Text"",""kwargs"":{""stream""...",threshold_crossing,-4.5 x RMS,NaN,NaN,NaN,...,Card2024Brain2Text,2023-08-11,25,train,2,1.0,attempted,True,,,NaN,9.62,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
4,BandPower,0.0,9.62,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,spike_band_power,256,uV^2,z_scored,linear regression referencing per 64-electrode...,50.0,"{""cls"":""Card2024Brain2Text"",""kwargs"":{""stream""...",NaN,NaN,250.0,5000.0,NaN,...,Card2024Brain2Text,2023-08-11,25,train,2,1.0,attempted,True,,,NaN,9.62,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
5,Sentence,0.0,9.62,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,My family is closer.,...,Card2024Brain2Text,2023-08-11,25,train,2,1.0,attempted,True,,NaN,,9.62,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
6,SpikeCounts,0.0,9.60,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,threshold_crossings,256,counts/bin,z_scored,linear regression referencing per 64-electrode...,50.0,"{""cls"":""Card2024Brain2Text"",""kwargs"":{""stream""...",threshold_crossing,-4.5 x RMS,NaN,NaN,NaN,...,Card2024Brain2Text,2023-08-11,25,train,2,2.0,attempted,True,,,NaN,9.60,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
7,BandPower,0.0,9.60,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,spike_band_power,256,uV^2,z_scored,linear regression referencing per 64-electrode...,50.0,"{""cls"":""Card2024Brain2Text"",""kwargs"":{""stream""...",NaN,NaN,250.0,5000.0,NaN,...,Card2024Brain2Text,2023-08-11,25,train,2,2.0,attempted,True,,,NaN,9.60,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
8,Sentence,0.0,9.60,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t15.2023.08.11,,,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,What do they like?,...,Card2024Brain2Text,2023-08-11,25,train,2,2.0,attempted,True,,NaN,,9.60,card_2024,braingate2/T15,card_2024,NaN,NaN,NaN,NaN,NaN
9,SpikeCounts,0.0,10.04,"Card2024Brain2Text:block=2,date=2023-08-11,ori...",Card2024Brain2Text/T15,t

Read that table carefully, because the two members are shaped **very differently**:

- **card_2024** has 706 `SpikeCounts` events and 671 `Sentence` events. It stores one
  timeline *per trial*, each with its own short neural stream.
- **willet_2023** has 34 `SpikeCounts` events and 680 `Sentence` events. It stores one
  timeline *per recording block*, a continuous stream with 20 sentences inside it on these
  two days (40 on most later days).

A **timeline is not a trial**. It is whatever unit the source stored, and the two conventions are both normal. This is exactly why the `Segmenter` exists: it cuts windows out of a timeline against a trigger, so both layouts reduce to the same thing downstream.

In [5]:
per_member = events.groupby("collection_member").agg(
    timelines=("timeline", "nunique"),
    sessions=("session", "nunique"),
    sentences=("type", lambda s: (s == "Sentence").sum()),
)
per_member["sentences_per_timeline"] = (
    per_member.sentences / per_member.timelines).round(1)
print(per_member.to_string())

print("\nan events frame is rows, not trials:")
print(events[events.type == "Sentence"].head(3)[
    ["collection_member", "session", "start", "duration", "text"]].to_string(index=False))

                   timelines  sessions  sentences  sentences_per_timeline
collection_member                                                        
card_2024                706         2        671                     1.0
willet_2023               34         2        680                    20.0

an events frame is rows, not trials:
collection_member        session  start  duration                 text
        card_2024 t15.2023.08.11    0.0      6.42     Bring it closer.
        card_2024 t15.2023.08.11    0.0      9.62 My family is closer.
        card_2024 t15.2023.08.11    0.0      9.60   What do they like?


---
## 2. `StandardizeText` + `ValidateTarget` — normalize text, mark targets

Two small **generic** transforms, neither knowing anything about BIT. Together they replace the old one-class approach, and each is reusable by any pipeline.

**`StandardizeText`** writes a normalized copy of a text column. `scheme="v1"` keeps exactly
the characters the vocabulary can encode, so the two never drift apart. Rows with no text (a neural stream) get an empty string, not `NaN`, which makes the next step's `!= ''` well-defined.

**`ValidateTarget`** writes a boolean `target_is_valid` column from a **condition** — a pandas
expression evaluated as a mask (`df.eval`). It is the marking counterpart of neuralset's `QueryEvents`, which filters rows with the same grammar. It marks rather than drops because the segmenter needs the trigger row and the neural rows in one frame. `target_is_valid == True` is literally the query the `Segmenter` triggers on.

Because the condition is a general expression that can reference any column, a member needing a different rule just puts its own `ValidateTarget` in its chain — the `Chain` structure is the per-member mechanism, so neither transform carries a per-member dict.

> The cells below call `._run(events)` directly, to see each transform in isolation. In a real
> chain they are composed and `Chain.run()` calls them in order — section 12.

In [6]:
from neuralshed.transforms import StandardizeText, ValidateTarget, valid_targets

events = StandardizeText(source="text", dest="canonical_text", scheme="v1")._run(events)
events = ValidateTarget(condition="type == 'Sentence' and canonical_text != ''")._run(events)

targets = valid_targets(events)
print(f"{len(targets)} targets\n")
targets[["collection_member", "session", "text", "canonical_text", "target_is_valid"]].head(4)

1351 targets



,collection_member,session,text,canonical_text,target_is_valid
2,card_2024,t15.2023.08.11,Bring it closer.,bring it closer,True
5,card_2024,t15.2023.08.11,My family is closer.,my family is closer,True
8,card_2024,t15.2023.08.11,What do they like?,what do they like,True
11,card_2024,t15.2023.08.11,How is that good?,how is that good,True


`canonical_text` is lower-cased with punctuation stripped, while `text` keeps the source's original casing. Normalizing into the frame means what the model is scored against is visible.

### The label-free trials

`card_2024` ships a held-out test split: real neural recordings with the sentence label withheld. Those timelines carry `SpikeCounts` and `BandPower` but **no `Sentence` event at all**, so they never match the condition — no special case needed.

The `canonical_text != ''` clause is the second guard: a sentence whose text normalizes to empty is rejected too. An empty target trains the model toward the all-blank solution, and nothing raises.

In [7]:
card = events[events.collection_member == "card_2024"]
labelled = set(card[card.type == "Sentence"].timeline)

summary = pd.DataFrame({
    "timelines": card.groupby("original_split").timeline.nunique(),
    "with a Sentence": card[card.timeline.isin(labelled)].groupby(
        "original_split").timeline.nunique(),
}).fillna(0).astype(int)
summary["-> targets"] = summary["with a Sentence"]
print(summary.to_string())
print("\ncard's 35 test timelines have neural data but no label, so they never trigger.")

                timelines  with a Sentence  -> targets
original_split                                        
test                   35                0           0
train                 636              636         636
val                    35               35          35

card's 35 test timelines have neural data but no label, so they never trigger.


---
## 3. `AssignHashSplit` — train, val, test

Splits are assigned **before** anything is fitted, because normalization statistics must come from training trials only and can't if the split does not exist yet.

The subtlety is *what counts as a trial*, and section 1 already showed the two members disagree:

- **card_2024** stores one timeline per trial, so the timeline identifies the trial.
- **willet_2023** stores one timeline per recording block, with 20 to 50 sentences inside it.
  Here the trial is the `Sentence` event itself.

neuralset's `SklearnSplit(split_by="timeline")` splits *whole timelines*, which is right for card and coarse for willet: a whole block lands in one bucket, so its sentences never spread over train and val.

`AssignHashSplit` keys on a **composite** of columns instead. `by=["timeline", "start"]` handles both without a per-member rule:

| member | trial rows | key | effect |
|---|---|---|---|
| card | all at `start=0.0` | `timeline ⎸0.0` | identical for a trial's rows, so the trial stays intact |
| willet | each at its own start | `timeline ⎸8.0`, `timeline ⎸25.86`, ... | each sentence splits independently |

It hashes the key with neuralset's own `DeterministicSplitter`, so the assignment is stable across runs, independent of row order, and unchanged when you later add a dataset to the collection. That last property matters: a shuffle-based splitter would reassign card's trials the moment willet joined, silently invalidating comparisons against earlier runs.

In [8]:
from neuralshed.transforms import AssignHashSplit

events = AssignHashSplit(by=["timeline", "start"],
                         ratios={"train": 0.8, "val": 0.1, "test": 0.1})._run(events)

print(valid_targets(events).groupby(["collection_member", "split"]).size().to_string())

collection_member  split
card_2024          test      82
                   train    537
                   val       52
willet_2023        test      73
                   train    529
                   val       78


Both members now spread across all three splits. Compare against `by=["timeline"]`, which reproduces `SklearnSplit`'s behaviour and puts each willet block wholly in one bucket:

In [9]:
coarse = AssignHashSplit(by=["timeline"])._run(events)
print("by=['timeline']  (whole-timeline, the SklearnSplit behaviour)")
print(valid_targets(coarse).groupby(["collection_member", "split"]).size().to_string())

by=['timeline']  (whole-timeline, the SklearnSplit behaviour)
collection_member  split
card_2024          test      67
                   train    540
                   val       64
willet_2023        test      40
                   train    580
                   val       60


Two caveats worth knowing, neither addressed here:

- **Repeated sentences.** willet reuses sentence texts across blocks, so the same text can land
  in train and val. Keying on `canonical_text` instead would fix it, at the cost of unbalanced
  split sizes.
- **Temporal adjacency.** Consecutive sentences are seconds apart in one recording, so a val
  trial sits between two train trials and shares slow neural drift with them. A time-blocked
  split is the conservative alternative.

### Capping trials for this notebook

`target_is_valid` is just a column, so trimming the working set is a filter, not a special mechanism. Twenty trials per (member, session, split) keeps extraction to a few seconds and a couple hundred megabytes. Drop this cell to use everything.

In [10]:
CAP = 20
keep = valid_targets(events).groupby(
    ["collection_member", "session", "split"], group_keys=False).head(CAP).index
events["target_is_valid"] = events.index.isin(keep)

print(f"capped to {int(events.target_is_valid.sum())} targets")
print(valid_targets(events).groupby(["collection_member", "session", "split"]).size().to_string())

capped to 236 targets
collection_member  session         split
card_2024          t15.2023.08.11  test     20
                                   train    20
                                   val      16
                   t15.2023.08.13  test     20
                                   train    20
                                   val      20
willet_2023        t12.2022.04.28  test     20
                                   train    20
                                   val      20
                   t12.2022.05.05  test     20
                                   train    20
                                   val      20


---
## 4. `CollectionManifest` — what the model needs to know about the recordings

The neuralshed database records one channel count and one bin rate per dataset, documented as *the predominant stored layout*. That is a catalog summary, not a contract: `jude_2026_speech` runs 256 and 384 channels with a **different column-to-area mapping** between them, and `kunz_2025` spans 128 and 256 channels and four bin sizes inside one Study. So the manifest is resolved by reading the events the collection actually produced.

It holds **one row per stream**, not per recording, because channel count and rate are properties of a stream:

| column | meaning |
|---|---|
| `layout_key` | which read-in projection handles this stream — it *is* the layout |
| `group_key` | which recording it belongs to, so a trial can assemble its streams |
| `event_type`, `kind`, `role` | the stream, its shape (`data` / `text`), and whether it is read or decoded |
| `channels`, `native_rate` | empty for anything that is not a dense array |
| `participant`, `region` | who and where, both shared across recordings |

A member with no band power simply has no such row. Absence is the representation, so a missing stream can never be mistaken for a block of real zeros.

In [11]:
from neuralshed.manifest import CollectionManifest

# One row per (recording, stream). `inputs` are wanted-if-present, `targets` are required, and
# both are neuroai event class names -- the vocabulary the catalog already uses.
manifest = CollectionManifest.from_events(
    events,
    inputs=("SpikeCounts", "BandPower"),
    targets=("Sentence",),
    participants={"card_2024": "braingate2/T15", "willet_2023": "braingate2/T12"},
    regions={"card_2024": "speech_precentral", "willet_2023": "speech_precentral"},
)
print(manifest, "\n")
manifest.to_frame()[
    ["layout_key", "group_key", "event_type", "kind", "role", "channels", "native_rate"]
]

CollectionManifest(12 layouts over 4 recordings, 2 participants, inputs=['BandPower', 'SpikeCounts'], targets=['Sentence']) 



,layout_key,group_key,event_type,kind,role,channels,native_rate
0,braingate2/T12/Willett2023Sentences/t12.2022.0...,willet_2023/t12.2022.04.28,BandPower,data,input,256.0,50.0
1,braingate2/T12/Willett2023Sentences/t12.2022.0...,willet_2023/t12.2022.04.28,Sentence,text,target,NaN,NaN
2,braingate2/T12/Willett2023Sentences/t12.2022.0...,willet_2023/t12.2022.04.28,SpikeCounts,data,input,256.0,50.0
3,braingate2/T12/Willett2023Sentences/t12.2022.0...,willet_2023/t12.2022.05.05,BandPower,data,input,256.0,50.0
4,braingate2/T12/Willett2023Sentences/t12.2022.0...,willet_2023/t12.2022.05.05,Sentence,text,target,NaN,NaN
5,braingate2/T12/Willett2023Sentences/t12.2022.0...,willet_2023/t12.2022.05.05,SpikeCounts,data,input,256.0,50.0
6,braingate2/T15/Card2024Brain2Text/t15.2023.08....,card_2024/t15.2023.08.11,BandPower,data,input,256.0,50.0
7,braingate2/T15/Card2024Brain2Text/t15.2023.08....,card_2024/t15.2023.08.11,Sentence,text,target,NaN,NaN
8,braingate2/T15/Card2024Brain2Text/t15.2023.08....,card_2024/t15.2023.08.11,SpikeCounts,data,input,256.0,50.0
9,braingate2/T15/Card2024Brain2Text/t15.2023.08....,card_2024/t15.2023.08.13,BandPower,data,input,256.0,50.0


Twelve layouts over four recordings: two input streams and one target each. Two participants, **one** region — the region embedding is shared across everyone while the read-ins are not.

Both datasets store threshold crossings and spike-band power as separate 256-channel streams, which is where the `512` hard-coded in the upstream BIT config comes from. Here they stay separate rows, each with its own projection, and the model sums them.

### Layout granularity

The **key template** says how you want recordings grouped. Dropping `session` from it shares one projection across a participant's whole study: cheaper, and only safe when those recordings really do agree.

That is *checked*, not trusted. `conflicts()` reports any layout whose recordings disagree on channel count, rate or channel ordering, and the preflight fails on a non-empty report. Level 1 is uniform, so nothing conflicts — the cell below forges what `jude_2026_speech` actually looks like, a second session re-implanted at a different width, to show the check firing.

In [12]:
# The key states *intent*: how you want recordings grouped. Whether that grouping is *legal*
# is a separate question, answered by comparing what one projection cannot represent.
STREAMS = dict(
    inputs=("SpikeCounts", "BandPower"),
    targets=("Sentence",),
    participants={"card_2024": "braingate2/T15", "willet_2023": "braingate2/T12"},
)
PER_SESSION = ("participant", "study", "session", "event_type")
PER_STUDY   = ("participant", "study", "event_type")

for name, template in [("per session", PER_SESSION), ("per study", PER_STUDY)]:
    m = CollectionManifest.from_events(events, key_template=template, **STREAMS)
    print(f"{name:11s} -> {len(m):2d} layouts, conflicts: {m.conflicts() or 'none'}")

# forge a jude_2026_speech-style mismatch: one card session re-implanted at 384 channels
forged = events.copy()
odd = (forged.session == "t15.2023.08.13") & (forged.type == "SpikeCounts")
forged.loc[odd, "ch_names"] = ",".join(f"tx:{c}" for c in range(384))

print("\nwith one session at a different width, under a per-study key:")
for c in CollectionManifest.from_events(forged, key_template=PER_STUDY, **STREAMS).conflicts():
    print("  !", c)

per session -> 12 layouts, conflicts: none


per study   ->  6 layouts, conflicts: none

with one session at a different width, under a per-study key:


  ! layout 'braingate2/T15/Card2024Brain2Text/SpikeCounts' merges 2 recordings that disagree on channels ['256', '384']


---
## 5. Preflight — fail here, not at step 500

Every check catches a failure that is otherwise **silent**:

- a label-free trial trains against an empty target
- members at different rates train one of them at the wrong time scale
- a coarse layout key merges recordings where channel *k* means two different electrodes
- a character outside the vocabulary is dropped from the target without a word
- a trial shorter than its own target produces an infinite CTC loss that `zero_infinity=True` converts into a zero gradient

None of these raise on their own, and all of them leave the loss curve looking plausible.

It comes in two halves. `validate_manifest` reads the layout table alone, so it runs in milliseconds with no data drive mounted. `validate_collection_for_bit` adds the checks that need the events themselves: targets, vocabulary, splits, CTC feasibility.

In [13]:
from neuralshed.models.bit import validate_collection_for_bit, validate_manifest

# The structural half reads the layout table alone, so it runs offline against a stored
# manifest -- which is what a collection builder can show while members are still being picked.
print("structural (manifest only):"); print(validate_manifest(manifest), "\n")
print("full (adds targets, vocabulary, splits, CTC feasibility):")
print(validate_collection_for_bit(events, manifest=manifest, patch_bins=5))

structural (manifest only):
preflight OK
  layouts: 12
  recordings: 4
  participants: ['braingate2/T12', 'braingate2/T15']
  inputs: ['BandPower', 'SpikeCounts']
  targets: ['Sentence']
  regions: ['speech_precentral']
  rates (Hz): [50.0] 

full (adds targets, vocabulary, splits, CTC feasibility):
preflight OK
  events: 4191
  targets: ['Sentence']
  layouts: 12
  recordings: 4
  participants: ['braingate2/T12', 'braingate2/T15']
  inputs: ['BandPower', 'SpikeCounts']
  regions: ['speech_precentral']
  rates (Hz): [50.0]
  targets per member: {'willet_2023': 120, 'card_2024': 116}
  vocabulary: v1 (29 classes)
  split: {'train': 80, 'test': 80, 'val': 76}
  ctc feasibility: 236/236 trials


In [14]:
# what a broken collection looks like: one member resampled, the other not
broken = events.copy()
broken.loc[broken.collection_member == "willet_2023", "frequency"] = 100.0
report = validate_collection_for_bit(broken, patch_bins=5)
print("ok:", report.ok)
for e in report.errors:
    print("ERROR", e)

ok: False
ERROR layouts span [50.0, 100.0] Hz; declare a target rate on the collection, or the same patch duration means a different time span per member


---
## 6. Segmentation — from an events frame to arrays

Now the frame becomes tensors. One `Segmenter` per recording, because extractor output width is fixed by the recording.

```python
ns.dataloader.Segmenter(
    extractors={"SpikeCounts": NeuralFeatureExtractor(event_types="SpikeCounts", ...),
                "BandPower":   NeuralFeatureExtractor(event_types="BandPower", ...)},
    trigger_query="target_is_valid == True",   # the column ValidateTarget wrote
    duration=None,                             # each trial keeps its own length
)
```

This is where card's per-trial timelines and willet's per-block timelines stop mattering: both get cut into windows around their `Sentence` triggers.

- **`duration=None` means "use the trigger's own duration"**. Sentences run from about 3 to 24 seconds and CTC does not care, so there is no reason to force a common window.
- **Streams stay separate**, keyed by event class, rather than concatenated into one wide array. Concatenating would make a missing `BandPower` indistinguishable from a block of real zeros.

In [15]:
from neuralshed.models.bit.bundle import _extract_trials
from neuralshed.text import CharVocab

vocab = CharVocab.canonical()      # frozen charset, see section 7
PATCH_BINS = 5                     # the model's temporal stride, 100 ms at 50 Hz

t0 = time.time()
trials, dropped = _extract_trials(events, manifest, vocab,
                                  frequency="native", target_unit="Hz",   # the bundle's default: counts/bin -> Hz
                                  patch_bins=PATCH_BINS)
mb = sum(a.nbytes for t in trials for a in t.arrays.values()) / 1e6
print(f"{len(trials)} trials in {time.time() - t0:.1f}s, {mb:.0f} MB, dropped: {dropped}\n")

t = trials[0]
print(f"recording  {t.group_key}")
print(f"layouts    {t.layouts}")
print(f"arrays     {dict((k, v.shape) for k, v in t.arrays.items())}   (channels, bins)")
print(f"text       {t.text!r}")
print(f"target ids {t.ids[:12]} ...")
print(f"split      {t.split}")

lengths = np.array([t.n_frames for t in trials])
print(f"\nwindow length: {lengths.min()}-{lengths.max()} bins "
      f"({lengths.min()/50:.1f}-{lengths.max()/50:.1f} s), mean {lengths.mean():.0f}")

236 trials in 4.8s, 293 MB, dropped: {'no_target': 0, 'too_short': 0, 'empty_window': 0, 'no_signal': 0}

recording  card_2024/t15.2023.08.11
layouts    {'BandPower': 'braingate2/T15/Card2024Brain2Text/t15.2023.08.11/BandPower', 'SpikeCounts': 'braingate2/T15/Card2024Brain2Text/t15.2023.08.11/SpikeCounts'}
arrays     {'BandPower': (256, 321), 'SpikeCounts': (256, 321)}   (channels, bins)
text       'bring it closer'
target ids [ 4 20 11 16  9  1 11 22  1  5 14 17] ...
split      train

window length: 138-1236 bins (2.8-24.7 s), mean 606


### The CTC feasibility check

A trial is dropped when its emission length cannot cover its own target. The bound is
**not** `len(ids)`:

$$\text{minimum frames} = \text{len(ids)} + \text{number of adjacent repeats}$$

CTC needs a blank between two identical adjacent labels, so `"coffee"` costs one extra frame and `"all"` one more. Getting this wrong is not loud: with `zero_infinity=True` an infeasible trial contributes a **zero gradient** rather than an error, so it looks like training data and does nothing.

Nothing is dropped here — these sentences are long but the windows are longer.

In [16]:
from neuralshed.text import ctc_min_input_length, n_repeated_labels

for text in ["bring it closer", "coffee", "all", "aaa"]:
    ids = vocab.encode(text)
    print(f"{text:18s} len={len(ids):3d}  repeats={n_repeated_labels(ids)}  "
          f"min frames={ctc_min_input_length(ids)}")

margin = np.array([t.n_frames // PATCH_BINS - ctc_min_input_length(t.ids) for t in trials])
print(f"\nslack across the {len(trials)} real trials: "
      f"min {margin.min()} patches, median {int(np.median(margin))}")

bring it closer    len= 15  repeats=0  min frames=15
coffee             len=  6  repeats=2  min frames=8
all                len=  3  repeats=1  min frames=4
aaa                len=  3  repeats=2  min frames=5

slack across the 236 real trials: min 19 patches, median 72


---
## 7. The vocabulary

`CharVocab.canonical()` is a **frozen** charset: `a-z`, apostrophe, space, plus id 0 for the CTC blank. 29 classes, always.

The alternative — fitting the vocabulary to whatever texts a collection happens to contain — makes `n_classes` a property of the *data*, so a checkpoint trained on one collection has the wrong head width for the next, and the head's rows mean different characters. The frozen charset sidesteps that entirely.

`coverage()` reports characters a vocabulary cannot encode, which the preflight turns into an error rather than letting them vanish from the target.

In [17]:
print(vocab, "| blank id:", vocab.BLANK)
print("first ids:", list(vocab.stoi.items())[:6])
print("\nround-trip:", repr(vocab.decode(vocab.encode("Bring it closer!"))))
print("coverage over all real targets:", vocab.coverage(valid_targets(events).canonical_text))
print("coverage gap for a 2-char vocab:", CharVocab(list("ab")).coverage(["abc"]))

CharVocab(version='v1', n_classes=29) | blank id: 0
first ids: [(' ', 1), ("'", 2), ('a', 3), ('b', 4), ('c', 5), ('d', 6)]

round-trip: 'bring it closer'
coverage over all real targets: {}
coverage gap for a 2-char vocab: {'c': 1}


---
## 8. `BitPreprocessingState` — statistics fitted on training trials only

Electrode gains differ across arrays, sessions and people, so inputs need z-scoring. Two questions: *where the numbers live*, and *what they are computed from*.

**Where.** Policy is configuration (`NormalizationPolicy`, which serializes with the collection). The fitted values are a training artifact, registered as **buffers** on the read-in so they travel inside the checkpoint. Applying a stale or absent state then cannot happen quietly.

**What from.** `fit()` filters on `split == "train"`, and produces one entry per input layout.

`already_normalized` names `card_2024`, whose data arrives z-scored within session from source. It is passed in here rather than read off the manifest, because it is a decision somebody made — the manifest holds only what was resolved from the data.

In [18]:
from neuralshed.models.bit import BitPreprocessingState, NormalizationPolicy

# `already_normalized` is a *declared* fact (card_2024 arrives z-scored from source), so it is
# passed in from the recipe rather than read off the manifest, which holds only resolved facts.
ALREADY_NORMALIZED = {"card_2024"}

state = BitPreprocessingState.fit(trials, manifest, NormalizationPolicy(),
                                  already_normalized=ALREADY_NORMALIZED)
print(state)
for key, s in state.stats.items():
    print(f"  {key:48s} mean{s.mean.shape} from {s.n_frames} frames")
print("\ncard layouts are absent above — declared already_normalized:", sorted(ALREADY_NORMALIZED))

BitPreprocessingState(4 entries, method='zscore', fitted_on=('train',))
  braingate2/T12/Willett2023Sentences/t12.2022.04.28/BandPower mean(256, 1) from 10335 frames
  braingate2/T12/Willett2023Sentences/t12.2022.04.28/SpikeCounts mean(256, 1) from 10335 frames
  braingate2/T12/Willett2023Sentences/t12.2022.05.05/BandPower mean(256, 1) from 10359 frames
  braingate2/T12/Willett2023Sentences/t12.2022.05.05/SpikeCounts mean(256, 1) from 10359 frames

card layouts are absent above — declared already_normalized: ['card_2024']


### Proving there is no leakage

`fingerprint()` is a digest of every fitted value. Fitting with the validation trials present and fitting without them must give the **identical** digest — direct evidence the fit never saw them.

In [19]:
train_only = [t for t in trials if t.split == "train"]
full = BitPreprocessingState.fit(trials, manifest, already_normalized=ALREADY_NORMALIZED).fingerprint()
held = BitPreprocessingState.fit(train_only, manifest, already_normalized=ALREADY_NORMALIZED).fingerprint()
print(f"fitted on all {len(trials)} trials:        {full[:16]}")
print(f"fitted on {len(train_only)} train trials only:  {held[:16]}")
print("identical:", full == held)

fitted on all 236 trials:        e791df310501f8d3
fitted on 80 train trials only:  e791df310501f8d3
identical: True


---
## 9. `collate_bit` — building a batch

Trials vary in length, so a batch is right-zero-padded to its longest item. Targets are right-padded with the blank, and `flat_targets` concatenates them into the form `nn.CTCLoss` expects.

**A batch must come from one recording.** Each stream is padded to a single channel count and routed through one projection, so `SessionBatchSampler(key_attr="group_key")` groups trials by recording and `collate_bit` raises if they are mixed. Checked rather than assumed, because a mixed batch trains perfectly happily on the wrong weights.

In [20]:
from neuralshed.models.bit import collate_bit, BitTrialDataset
from neuralshed.models.data import SessionBatchSampler
from torch.utils.data import DataLoader

first_group = trials[0].group_key
same = [t for t in trials if t.group_key == first_group][:4]
batch = collate_bit(same)

print("recording     ", batch.group_key)
print("layouts       ", batch.layouts)
print("arrays        ", {k: tuple(v.shape) for k, v in batch.arrays.items()}, " (B, C, T)")
print("input_lengths ", batch.input_lengths.tolist(), " <- padded to the longest")
print("target_ids    ", tuple(batch.target_ids.shape), "(B, S), blank-padded")
print("target_lengths", batch.target_lengths.tolist())
print("flat_targets  ", tuple(batch.flat_targets.shape))

other = next(t for t in trials if t.group_key != first_group)
try:
    collate_bit([same[0], other])
except ValueError as e:
    print("\nmixing recordings:", e)

recording      card_2024/t15.2023.08.11
layouts        {'BandPower': 'braingate2/T15/Card2024Brain2Text/t15.2023.08.11/BandPower', 'SpikeCounts': 'braingate2/T15/Card2024Brain2Text/t15.2023.08.11/SpikeCounts'}
arrays         {'BandPower': (4, 256, 502), 'SpikeCounts': (4, 256, 502)}  (B, C, T)
input_lengths  [321, 481, 480, 502]  <- padded to the longest
target_ids     (4, 19) (B, S), blank-padded
target_lengths [15, 19, 17, 16]
flat_targets   (67,)

mixing recordings: a batch must come from one recording, got ['card_2024/t15.2023.08.11', 'card_2024/t15.2023.08.13']


---
## 10. The model

```
SpikeCounts (B, C_spike, T) ─ Linear(C_spike → D) ─┐
BandPower   (B, C_sbp,   T) ─ Linear(C_sbp   → D) ─┴─ sum ─→ (B, T, D)   BitReadIn
                                     + participant emb + region emb
                                              │
                       Conv1d(D → D, kernel = stride = patch_bins)         BitPatcher
                                              │  (B, P, D)
                       transformer, causal mask AND padding mask           BitTrunk
                                              │
                       Linear(D → n_classes)                               CTC head
```

One projection per layout, and one layout is one stream, so which projection a tensor goes through is decided by the manifest rather than by a position in a list.

### Why the read-in runs before the patcher

Upstream BIT does the reverse: smooth, patchify, *then* embed per dataset. Both the smoother and the patchifier are built around a fixed width and run before any per-dataset layer sees the tensor, so the `512` in the upstream config is a channel count masquerading as a hyper-parameter.

Moving the read-in first has three consequences:

1. Raw channel count reaches **only** `BitReadIn`. Adding a recording adds one projection per stream.
2. Projections become `C → D` instead of `5C → D`, roughly a fifth of the parameters.
3. The patcher becomes a plain strided conv over the shared width.

### The patch is stated in milliseconds

`patch_ms`, not `patch_bins`. 100 ms is 5 bins at 50 Hz and 10 bins at 100 Hz, so the same config text describes both and the trunk always sees the same physical span.

In [21]:
from neuralshed.models.bit import BitConfig, build_bit_model

config = BitConfig.from_manifest(manifest, vocab.n_classes, patch_ms=100.0)
print(f"patch_ms={config.patch_ms} at {config.frequency} Hz -> patch_bins={config.patch_bins}")

model = build_bit_model(manifest, vocab, state,
                        patch_ms=100.0, dim=128, depth=2, heads=4, dim_head=32, dropout=0.1)
print("parameters:", model.n_parameters())
print("input layouts:", list(model.readin.layout_keys))

patch_ms=100.0 at 50.0 Hz -> patch_bins=5
parameters: {'readin': 263808, 'shared': 483003, 'total': 746811}
input layouts: ['braingate2/T12/Willett2023Sentences/t12.2022.04.28/BandPower', 'braingate2/T12/Willett2023Sentences/t12.2022.04.28/SpikeCounts', 'braingate2/T12/Willett2023Sentences/t12.2022.05.05/BandPower', 'braingate2/T12/Willett2023Sentences/t12.2022.05.05/SpikeCounts', 'braingate2/T15/Card2024Brain2Text/t15.2023.08.11/BandPower', 'braingate2/T15/Card2024Brain2Text/t15.2023.08.11/SpikeCounts', 'braingate2/T15/Card2024Brain2Text/t15.2023.08.13/BandPower', 'braingate2/T15/Card2024Brain2Text/t15.2023.08.13/SpikeCounts']


The read-in is a minority of the parameters; everything else is shared across all four recordings and both participants. That ratio is the point of pooling.

### Shapes through a forward pass

In [22]:
model.eval()
with torch.no_grad():
    h = model.readin(batch.arrays, batch.layouts, batch.participant_idx, batch.region_idx)
    x = model.patcher(h)
    logits = model(batch.as_dict())
lengths = model.out_lengths(batch.input_lengths)

for stream, tensor in batch.arrays.items():
    print(f"input   {tuple(tensor.shape)}  (B, C, T)   {stream}, summed after its own Linear")
print(f"read-in {tuple(h.shape)}  (B, T, D)   <- channel count absorbed here")
print(f"patcher {tuple(x.shape)}   (B, P, D)   <- P = ceil(T / {config.patch_bins})")
print(f"logits  {tuple(logits.shape)}    (B, P, n_classes)")
print(f"\nemission lengths: {lengths.tolist()}")

input   (4, 256, 502)  (B, C, T)   BandPower, summed after its own Linear
input   (4, 256, 502)  (B, C, T)   SpikeCounts, summed after its own Linear
read-in (4, 502, 128)  (B, T, D)   <- channel count absorbed here
patcher (4, 101, 128)   (B, P, D)   <- P = ceil(T / 5)
logits  (4, 101, 29)    (B, P, n_classes)

emission lengths: [65, 97, 96, 101]


### Flexible width, one trunk

Level 1 is uniform: eight input layouts, every one of them 256 channels. The width flexibility only gets exercised at level 3, when `fan_2023` arrives at 192 channels with **no** spike-band power.

The machinery already handles it, which is worth seeing. Below, one recording with two 256-channel streams and one with a single 192-channel stream — `fan_2023`'s real shape — under one shared trunk. No model edit, no config surgery.

In [23]:
from neuralshed.models.bit import BitCrossEncoder

# One layout per stream, so widths are a flat table and a member with no band power simply has
# no second entry. Nothing about the trunk changes between the two.
mixed = BitCrossEncoder(BitConfig(
    n_classes=vocab.n_classes,
    layout_keys=("T15/speech/SpikeCounts", "T15/speech/BandPower", "T5/handknob/SpikeCounts"),
    layout_channels={"T15/speech/SpikeCounts": 256, "T15/speech/BandPower": 256,
                     "T5/handknob/SpikeCounts": 192},          # fan_2023's real shape
    layout_streams={"T15/speech/SpikeCounts": "SpikeCounts", "T15/speech/BandPower": "BandPower",
                    "T5/handknob/SpikeCounts": "SpikeCounts"},
    n_participants=2, n_regions=2, frequency=50.0, patch_ms=100.0,
    dim=64, depth=1, heads=2, dim_head=16,
)).eval()

def probe(layouts, arrays):
    with torch.no_grad():
        return mixed({"arrays": arrays, "layouts": layouts,
                      "input_lengths": torch.tensor([300] * 2),
                      "participant_idx": torch.zeros(2, dtype=torch.long),
                      "region_idx": torch.zeros(2, dtype=torch.long)}).shape

print("256ch + band power ->", tuple(probe(
    {"SpikeCounts": "T15/speech/SpikeCounts", "BandPower": "T15/speech/BandPower"},
    {"SpikeCounts": torch.randn(2, 256, 300), "BandPower": torch.randn(2, 256, 300)})))
print("192ch, no band power ->", tuple(probe(
    {"SpikeCounts": "T5/handknob/SpikeCounts"},
    {"SpikeCounts": torch.randn(2, 192, 300)})))
print("\nsame trunk, same output width: raw channel count never reaches it")


256ch + band power -> (2, 60, 29)
192ch, no band power -> (2, 60, 29)

same trunk, same output width: raw channel count never reaches it


### The padding mask

Upstream BIT's only attention mask is the causal one. Its own config comment recommends `causal=False` for offline accuracy — at which point **nothing masks padding** and every query attends freely to zero-filled frames. A batch's results then depend on what else happened to be batched with it.

Here `BitTrunk` always takes a key-padding mask. The check: a padded batch and the same item alone must give identical logits on the valid frames. These are real trials of genuinely different lengths, so the padding is substantial.

In [24]:
pool = [t for t in trials if t.group_key == first_group]
short, long = min(pool, key=lambda t: t.n_frames), max(pool, key=lambda t: t.n_frames)
print(f"short trial {short.n_frames} bins, long trial {long.n_frames} bins "
      f"-> {long.n_frames - short.n_frames} bins of padding")

with torch.no_grad():
    padded = model(collate_bit([short, long]).as_dict())
    alone = model(collate_bit([short]).as_dict())

n_valid = int(model.out_lengths(torch.tensor([short.n_frames])))
delta = (padded[0, :n_valid] - alone[0, :n_valid]).abs().max().item()
print(f"max logit difference over {n_valid} valid patches: {delta:.2e}  -> identical: {delta < 1e-5}")

short trial 138 bins, long trial 902 bins -> 764 bins of padding
max logit difference over 28 valid patches: 9.54e-07  -> identical: True


### Unknown layouts raise

Upstream routed anything unmatched through `dataset_keys[0]`, so a misconfigured run trained on the wrong weights and reported nothing at all.

In [25]:
unknown = {**batch.layouts, "SpikeCounts": "braingate2/T12/willet/1999.01.01/SpikeCounts"}
try:
    model(batch.as_dict() | {"layouts": unknown})
except KeyError as e:
    print("KeyError:", e)

KeyError: "no projection for layout 'braingate2/T12/willet/1999.01.01/SpikeCounts'; the model was built from a manifest with 8 input layouts and this is not one"


---
## 11. Training

CTC loss over `(P, B, n_classes)` log-probabilities, with `out_lengths()` giving the emission length per item — the *same* arithmetic the patcher used, so CTC never sees a length the trunk did not produce.

`SessionBatchSampler` interleaves recordings across the epoch, so consecutive steps come from different sessions and the shared trunk sees a mix rather than one recording at a time.

A few epochs on 80 trials will not produce a usable decoder — real runs are tens of thousands of trials and tens of epochs. What it does show is that the contract holds on real data: finite loss, finite gradients, and loss falling.

In [26]:
model.train()
ctc = torch.nn.CTCLoss(blank=vocab.BLANK, zero_infinity=True)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

train_ds = BitTrialDataset(trials, split="train")
loader = DataLoader(train_ds, collate_fn=collate_bit,
                    batch_sampler=SessionBatchSampler(train_ds, batch_size=4,
                                                      key_attr="group_key"))
print(f"{len(train_ds)} training trials, {len(loader)} batches per epoch\n")

for epoch in range(6):
    losses = []
    for b in loader:
        log_probs = model(b.as_dict())
        loss = ctc(log_probs.permute(1, 0, 2), b.flat_targets,
                   model.out_lengths(b.input_lengths), b.target_lengths)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
    print(f"epoch {epoch}  mean CTC loss {np.mean(losses):7.3f}")

80 training trials, 20 batches per epoch



epoch 0  mean CTC loss   5.623


epoch 1  mean CTC loss   4.143


epoch 2  mean CTC loss   4.050


epoch 3  mean CTC loss   3.762


epoch 4  mean CTC loss   3.597


epoch 5  mean CTC loss   3.554


### Decoding

Greedy CTC decode: argmax, collapse repeats, drop blanks. After six epochs on 80 trials the model has learned roughly the letter frequencies of English and nothing else, so expect a CER near 100%. The point is that the decode path and the metric work end to end.

A CER above 100% would not be a bug either. It is edit distance divided by *reference* length, so a hypothesis longer than the reference can need more edits than the reference has characters.

Note the validation batch comes from the **sampler**, not from slicing the trial list. Slicing would mix recordings and `collate_bit` would refuse it.

In [27]:
from neuralshed.models.training import greedy_decode, char_error_rate

model.eval()
val_ds = BitTrialDataset(trials, split="val")
val_loader = DataLoader(val_ds, collate_fn=collate_bit,
                        batch_sampler=SessionBatchSampler(val_ds, batch_size=4, shuffle=False,
                                                          key_attr="group_key"))

preds, refs = [], []
with torch.no_grad():
    for b in val_loader:
        preds += greedy_decode(model(b.as_dict()), model.out_lengths(b.input_lengths), vocab)
        refs += b.texts

for p, r in list(zip(preds, refs))[:3]:
    print(f"  ref {r!r}\n  hyp {p!r}\n")
print(f"CER over {len(refs)} validation trials: {char_error_rate(preds, refs):.1f}%")

  ref 'is my nurse okay'
  hyp 'sh yhtehoh e e eoyheyo '

  ref 'i am that good'
  hyp 't tyot ht n et e '

  ref 'tell my family i am thirsty'
  hyp 'hoty oto eo e e o ot'

CER over 76 validation trials: 88.2%


---
## 12. All of it, in one call

Everything above is what `BitDataBundle.from_collection` does. It accepts either a saved collection name or a runnable `ns.Step`, so the in-memory chain works directly:

In [28]:
from neuralshed.models.bit import BitDataBundle
from neuralshed.transforms import StandardizeText, ValidateTarget, AssignHashSplit

chain = ns.Chain(steps={
    "CombineStudies": transforms.CombineStudies(
        chains={"card_2024": CARD, "willet_2023": WILLET}),
    "StandardizeText": StandardizeText(),
    "ValidateTarget": ValidateTarget(),
    "AssignHashSplit": AssignHashSplit(by=["timeline", "start"]),
})

bundle = BitDataBundle.from_collection(
    chain,
    inputs=("SpikeCounts", "BandPower"),
    targets=("Sentence",),
    participants={"card_2024": "braingate2/T15", "willet_2023": "braingate2/T12"},
    regions={"card_2024": "speech_precentral", "willet_2023": "speech_precentral"},
    already_normalized={"card_2024"},
    verbose=False,
)
print(bundle)
print("dropped:", bundle.dropped)

BitDataBundle(1351 trials {'test': 155, 'train': 1066, 'val': 130}, 12 layouts, n_classes=29)
dropped: {'no_target': 0, 'too_short': 0, 'empty_window': 0, 'no_signal': 0}


That is the uncapped set — every trial from the four sessions, which is why the count is far larger than the 80 used above.

From there a full run is:

```python
from neuralshed.models.training import CTCModule
import lightning as L

model = build_bit_model(bundle.manifest, bundle.vocab, bundle.state)
L.Trainer(max_epochs=30).fit(CTCModule(model, bundle.vocab),
                             bundle.train_loader(), bundle.val_loader())
```

### Persisting the collection

To load it by name instead, save the same chain to the database. A collection also declares **what it is for** — which event classes it reads, and which it decodes:

```python
dbc.save_collection(
    collection_name="speech-text",
    members=["card_2024", "willet_2023"],
    input_event_types=("SpikeCounts", "BandPower"),   # wanted-if-present
    target_event_types=("Sentence",),                 # required
    chain={"name": "Chain", "steps": {
        "CombineStudies": {"name": "CombineStudies", "chains": {
            "card_2024":   {"name": "Card2024Brain2Text"},
            "willet_2023": {"name": "Willett2023Sentences"},
        }},
        "StandardizeText": {"name": "StandardizeText"},
        "ValidateTarget": {"name": "ValidateTarget"},
        "AssignHashSplit": {"name": "AssignHashSplit", "by": ["timeline", "start"]},
    }},
)
```

Those two lists are checked against the catalog as you save, with no data drive mounted, so a member that cannot supply the target is refused there and then rather than showing up later as an empty extraction.

Saving is cheap. Resolving the layout table is the slow part — it runs the chain and opens data files — so it is a separate, re-callable step:

```python
dbc.manifest_status(collection_name="speech-text")   # not_built | stale | built
dbc.build_manifest(collection_name="speech-text")    # ~2 min, then stored
bundle = BitDataBundle.from_collection("speech-text")
```

A member needing a *different* rule (a stricter target filter, a source-published split) puts its own transform in that member's chain — `[Card2024Brain2Text, ValidateTarget(...)]` — rather than a per-member dict on the shared step. The `Chain` structure is the per-member mechanism.

The stored config carries no `path`: data directories are injected at load time, so the same JSON works on any machine with the drive mounted.

---
## Where this stops

This is **collection level 1**: two members, two participants, one region, both natively 50 Hz, both 256 + 256 channels. Nothing varies but the person. Widening adds one kind of heterogeneity at a time, so a failure isolates to the thing just added:

| level | adds | needs |
|---|---|---|
| 2 | wairagkar_2025 | 100 Hz — modality-aware resampling (count-preserving for spikes), Gaussian smoothing as a wrapper extractor, cross-dataset participant identity for T15 |
| 3 | fan_2023 | 192 channels and **no** band power — the read-in's missing-stream path, previewed in section 10 |
| 4 | willet_2021 | 10 ms bins, sentences mixed with single characters |
| 5 | jude_2026_speech, kunz_2025 | layout varying *within* one Study — where per-session layout keys stop being optional |
| 6 | productization | recipe persisted on the collection, materialized extraction cache, phoneme targets |

The plan for each is in [`others/plan_bit_ready_collections.md`](../../../others/plan_bit_ready_collections.md), with the neuroai API details in [`others/plan_bit_api_notes.md`](../../../others/plan_bit_api_notes.md).